# ComfyUI on Kaggle

这是一份全新 notebook（不修改你原有文件），用于在 Kaggle 里运行 ComfyUI。

按顺序执行下面代码单元：
1. 克隆 ComfyUI
2. 安装 ComfyUI Manager
3. 安装缺失的自定义节点
4. 安装依赖
5. 下载模型
6. 启动 ComfyUI + Ngrok
7. 故障排查：查看日志与端口
8. 保活（可选）

注意：Kaggle Session 重启后需要重新运行。

In [ ]:
# 1) 克隆 ComfyUI

%cd /kaggle/
!git clone https://github.com/comfyanonymous/ComfyUI.git

print("\nComfyUI 克隆完成")

In [ ]:
# 2) 克隆 ComfyUI-Manager

%cd /kaggle/ComfyUI/custom_nodes
!git clone https://github.com/ltdrdata/ComfyUI-Manager.git

In [ ]:
# 3) 安装其他社区节点和依赖

import os, re, shutil, subprocess, sys, tempfile, venv

CUSTOM_NODES = "/kaggle/ComfyUI/custom_nodes"
VENV_DIR = "/kaggle/working/comfyui-venv"
RUNTIME_PYTHON = f"{VENV_DIR}/bin/python"

# Avoid interactive git prompts in Kaggle
os.environ["GIT_TERMINAL_PROMPT"] = "0"
subprocess.call(["git", "config", "--global", "credential.helper", ""])

# Run venv Python with a sanitized environment to avoid Kaggle sitecustomize issues
CLEAN_ENV = os.environ.copy()
CLEAN_ENV.pop("PYTHONPATH", None)
CLEAN_ENV.pop("PYTHONHOME", None)
# CLEAN_ENV["PYTHONNOUSERSITE"] = "1"  # Removed to allow wrapt import

if not os.path.exists(RUNTIME_PYTHON):
    print("[setup] Creating ComfyUI venv for node dependencies...")
    try:
        venv.create(VENV_DIR, with_pip=True)
    except Exception:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "virtualenv"])
        subprocess.check_call([sys.executable, "-m", "virtualenv", VENV_DIR])

# Bootstrap pip tooling and wrapt inside the runtime venv first
subprocess.check_call([RUNTIME_PYTHON, "-m", "pip", "install", "-q", "-U", "pip", "setuptools", "wheel", "wrapt"], env=CLEAN_ENV)

repos = [
    ("ComfyUI-WanVideoWrapper",        "https://github.com/kijai/ComfyUI-WanVideoWrapper.git"),
    ("comfyui_controlnet_aux",         "https://github.com/Fannovel16/comfyui_controlnet_aux.git"),
    ("ComfyUI-KJNodes",                "https://github.com/kijai/ComfyUI-KJNodes.git"),
    ("ComfyUI-Easy-Use",               "https://github.com/yolain/ComfyUI-Easy-Use.git"),
    ("ComfyUI-VideoHelperSuite",       "https://github.com/Kosinkadink/ComfyUI-VideoHelperSuite.git"),
    ("ComfyUI_essentials",             "https://github.com/cubiq/ComfyUI_essentials.git"),
    ("audio-separation-nodes-comfyui", "https://github.com/christian-byrne/audio-separation-nodes-comfyui.git"),
    ("comfyui-various",                "https://github.com/jamesWalker55/comfyui-various.git"),
]

def _owner_repo(url):
    m = re.search(r"github.com/([^/]+)/([^/.]+)(?:\.git)?$", url)
    if not m:
        return None, None
    return m.group(1), m.group(2)

def install_repo(name, url, dest):
    if os.path.isdir(dest) and os.listdir(dest):
        print(f"[skip] {name} already exists")
        return True

    print(f"[clone] {name}")
    r = subprocess.run(["git", "clone", "--depth=1", url, dest], env=CLEAN_ENV)
    if r.returncode == 0:
        return True

    print(f"[warn] git clone failed for {name}, trying zip fallback")
    owner, repo = _owner_repo(url)
    if not owner:
        return False

    with tempfile.TemporaryDirectory() as td:
        zip_path = os.path.join(td, f"{repo}.zip")
        for branch in ("main", "master"):
            zip_url = f"https://codeload.github.com/{owner}/{repo}/zip/refs/heads/{branch}"
            d = subprocess.run(["wget", "-q", "-O", zip_path, zip_url], env=CLEAN_ENV)
            if d.returncode != 0 or (not os.path.exists(zip_path)) or os.path.getsize(zip_path) == 0:
                continue

            u = subprocess.run(["unzip", "-q", zip_path, "-d", td], env=CLEAN_ENV)
            if u.returncode != 0:
                continue

            extracted = os.path.join(td, f"{repo}-{branch}")
            if os.path.isdir(extracted):
                if os.path.exists(dest):
                    shutil.rmtree(dest)
                shutil.move(extracted, dest)
                print(f"[ok] Installed {name} via zip fallback")
                return True

    return False

clone_failed = []
for name, url in repos:
    dest = os.path.join(CUSTOM_NODES, name)
    ok = install_repo(name, url, dest)
    if not ok:
        clone_failed.append(name)

pip_failed = []
for name, _ in repos:
    if name in clone_failed:
        continue
    req = os.path.join(CUSTOM_NODES, name, "requirements.txt")
    if os.path.exists(req):
        print(f"[pip] installing requirements for {name}")
        pr = subprocess.run([RUNTIME_PYTHON, "-m", "pip", "install", "-q", "--disable-pip-version-check", "-r", req], env=CLEAN_ENV)
        if pr.returncode != 0:
            pip_failed.append(name)

if clone_failed or pip_failed:
    print("\n[summary] Some nodes were not fully prepared")
    if clone_failed:
        print(f"  clone failed: {clone_failed}")
    if pip_failed:
        print(f"  requirements failed: {pip_failed}")
    print("Continuing with available nodes...")
else:
    print("\n[done] All custom nodes installed and dependencies prepared.")

In [ ]:
# 4) 设置 venv 并安装 ComfyUI 依赖

import os, sys, subprocess, venv

VENV_DIR = "/kaggle/working/comfyui-venv"
PYTHON = f"{VENV_DIR}/bin/python"

if not os.path.exists(PYTHON):
    print("Creating isolated venv for ComfyUI...")
    try:
        venv.create(VENV_DIR, with_pip=True)
    except Exception as e:
        print(f"venv failed: {e}. Falling back to virtualenv...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "virtualenv"])
        subprocess.check_call([sys.executable, "-m", "virtualenv", VENV_DIR])

subprocess.check_call([PYTHON, "-m", "pip", "install", "-U", "pip", "setuptools", "wheel"])
subprocess.check_call([PYTHON, "-m", "pip", "install", "-r", "/kaggle/ComfyUI/requirements.txt"])
print(f"ComfyUI Python: {PYTHON}")

In [ ]:
# 5） 下载模型

import os
import concurrent.futures
import subprocess

# 确保目录存在
os.makedirs("/kaggle/ComfyUI/models/checkpoints", exist_ok=True)
os.makedirs("/kaggle/ComfyUI/models/loras", exist_ok=True)
os.makedirs("/kaggle/ComfyUI/models/vae", exist_ok=True)
os.makedirs("/kaggle/ComfyUI/models/clip_vision", exist_ok=True)
os.makedirs("/kaggle/ComfyUI/models/text_encoders", exist_ok=True)
os.makedirs("/kaggle/ComfyUI/models/llm", exist_ok=True)
os.makedirs("/kaggle/ComfyUI/models/upscale_models", exist_ok=True)

def download_model(model_url, model_path):
    if os.path.exists(model_path):
        print(f"[skip] {os.path.basename(model_path)} already exists")
        return
    if model_url.startswith("URL_PLACEHOLDER"):
        print(f"[skip] 请补充下载链接：{os.path.basename(model_path)}")
        return
    print(f"[download] {os.path.basename(model_path)}")
    result = subprocess.run(["wget", "-q", "-O", model_path, model_url])
    if result.returncode == 0:
        print(f"[ok] Downloaded {os.path.basename(model_path)}")
    else:
        print(f"[fail] Failed to download {os.path.basename(model_path)}")

# 定义模型列表
models = [
    ("https://huggingface.co/adamo1139/stable-diffusion-3-medium-ungated/resolve/main/sd3_medium_incl_clips_t5xxlfp8.safetensors?download=true", "/kaggle/ComfyUI/models/checkpoints/sd3_medium_incl_clips_t5xxlfp8.safetensors"),
    ("https://huggingface.co/stabilityai/stable-diffusion-xl-base-1.0/resolve/main/sd_xl_base_1.0.safetensors", "/kaggle/ComfyUI/models/checkpoints/sd_xl_base_1.0.safetensors"),  
    ("https://civitai.com/api/download/models/309729?type=VAE&format=SafeTensor", "/kaggle/ComfyUI/models/vae/sdxl_vae.safetensors"),
    ("https://huggingface.co/h94/IP-Adapter/resolve/main/sdxl_models/image_encoder/model.safetensors", "/kaggle/ComfyUI/models/clip_vision/clip_vision_h.safetensors"),
    ("https://huggingface.co/ai-forever/Real-ESRGAN/resolve/main/RealESRGAN_x2.pth", "/kaggle/ComfyUI/models/upscale_models/RealESRGAN_x2.pth"),
    ("https://huggingface.co/MeiGen-AI/InfiniteTalk/resolve/main/comfyui/infinitetalk_single.safetensors", "/kaggle/ComfyUI/models/checkpoints/infinitetalk_single.safetensors"),
    ("https://huggingface.co/Kijai/WanVideo_comfy/resolve/main/Wan2_1_VAE_bf16.safetensors", "/kaggle/ComfyUI/models/vae/Wan2_1_VAE_bf16.safetensors"),
    ("https://huggingface.co/Kijai/WanVideo_comfy/resolve/main/umt5-xxl-enc-bf16.safetensors", "/kaggle/ComfyUI/models/text_encoders/umt5-xxl-enc-bf16.safetensors"),
    ("https://huggingface.co/Kijai/WanVideo_comfy/resolve/main/LoRAs/Wan22_relight/WanAnimate_relight_lora_fp16.safetensors", "/kaggle/ComfyUI/models/loras/Wan2.2AnimateWanAnimate_relight_lora_fp16.safetensors"),
    ("https://huggingface.co/Aitrepreneur/FLX/resolve/main/Wan2.2-Lightning_I2V-A14B-4steps-lora_LOW_fp16.safetensors", "/kaggle/ComfyUI/models/loras/Wan2.2AnimateWan2.2-Lightning_l2V-A14B-4steps-lora_LOW_fp16.safetensors"),
    ("https://huggingface.co/Kijai/WanVideo_comfy/resolve/main/FastWan/FastWan_T2V_14B_480p_lora_rank_128_bf16.safetensors", "/kaggle/ComfyUI/models/loras/Wan2.2AnimateFastWan_T2V_l4B_480p_lora_rank_128_bf16.safetensors"),
    ("https://huggingface.co/Kijai/WanVideo_comfy/resolve/main/Pusa/Wan21_PusaV1_LoRA_14B_rank512_bf16.safetensors", "/kaggle/ComfyUI/models/loras/Wan2.2AnimateWan21_PusaV1_LoRA_14B_rank512_bf16.safetensors"),
    ("https://huggingface.co/Kijai/WanVideo_comfy/resolve/ffc8175b07b79f430a1495d086e39e83d59729e0/Wan22_FunReward/Wan2.2-Fun-A14B-InP-LOW-HPS2.1_resized_dynamic_avg_rank_15_bf16.safetensors", "/kaggle/ComfyUI/models/loras/Wan2.2AnimateWan2.2-Fun-A14B-lnP-LOW-HPS2.1_resized_dynamic_avg_rank15_bf16.safetensors"),
    ("https://huggingface.co/QuantStack/Wan2.2-Animate-14B-GGUF/resolve/main/Wan2.2-Animate-14B-Q4_0.gguf", "/kaggle/ComfyUI/models/llm/Wan2.2-Animate-14B-Q4_0.gguf"),
]

# 并行下载
with concurrent.futures.ThreadPoolExecutor(max_workers=5) as executor:
    futures = [executor.submit(download_model, url, path) for url, path in models]
    for future in concurrent.futures.as_completed(futures):
        pass  # 等待所有完成

print("模型下载完成")



In [ ]:
# 6) 设置 ngrok 隧道并启动 ComfyUI

import os, sys, subprocess, gc
from kaggle_secrets import UserSecretsClient

VENV_DIR = "/kaggle/working/comfyui-venv"
PYTHON = f"{VENV_DIR}/bin/python"

# 获取 NGROK_TOKEN
try:
    user_secrets = UserSecretsClient()
    Ngrok_token = user_secrets.get_secret("NGROK_TOKEN")
    use_ngrok = True
except Exception as e:
    print(f"Error getting NGROK_AUTHTOKEN: {e}")
    print("跳过 ngrok，启动本地 ComfyUI")
    Ngrok_token = None
    use_ngrok = False
Ngrok_domain = ""  # optional, leave empty if you don't have a domain
port = 8188

# -----------------

if use_ngrok and not Ngrok_token:
    raise RuntimeError("Missing NGROK_AUTHTOKEN. Set it in Kaggle Secrets, then rerun.")

if use_ngrok:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "pyngrok==6.1.0"])

from pyngrok import ngrok, conf

gc.collect()

try:
    if use_ngrok:
        ngrok.set_auth_token(Ngrok_token)
        ngrok.kill()
        if Ngrok_domain:
            srv = ngrok.connect(port, domain=Ngrok_domain)
        else:
            srv = ngrok.connect(port)
        print(f"Ngrok ComfyUI URL: {srv.public_url}")
    else:
        print("本地 ComfyUI 将在 http://127.0.0.1:8188 启动")

    # Start ComfyUI using the isolated Python environment.
    subprocess.check_call([PYTHON, "/kaggle/ComfyUI/main.py"])
except Exception as e:
    print(f"Error starting: {e}")

In [ ]:
# 7) 故障排查：查看日志与端口
!echo "===== ComfyUI 进程 ====="
!ps -ef | grep -i "comfyui\|main.py" | grep -v grep || true
!echo "===== 监听端口 (8188) ====="
!ss -lntp | grep 8188 || true
!echo "===== ngrok 隧道 ====="
!curl -s http://127.0.0.1:4040/api/tunnels | python -m json.tool || echo "ngrok not running or API not accessible"

In [ ]:
# 8) 保活（可选）
import time
import urllib.request

print(f"Keepalive started. Checking ComfyUI on port 8188 every 30s")
print("Press Interrupt to stop this cell.")

while True:
    try:
        with urllib.request.urlopen("http://127.0.0.1:8188", timeout=5) as resp:
            print(time.strftime("%H:%M:%S"), "alive", "status=", resp.status)
    except Exception as e:
        print(time.strftime("%H:%M:%S"), "check_failed", str(e))
    time.sleep(30)